In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import lit
#Stream from a csv file

schema = StructType([
    StructField("ID", IntegerType(), True),
    StructField("NAME", StringType(), True),
 
])

'''stream_df = spark.readStream.format("csv").option("checkpointLocation","/Volumes/initialcatalog/initialschema/checkpointvolume/checkpoint").option("header", "true").option("basePath","/Volumes/initialcatalog/initialschema/initialvolume/streamfolder/").schema(schema).load("/Volumes/initialcatalog/initialschema/initialvolume/streamfolder/Student.csv").withColumn("Age", lit(22)).select("ID", "NAME", "Age")
'''
chkpointpath = "/Volumes/initialcatalog/initialschema/checkpointvolume/write_checkpoint"
stream_df = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .option("delimiter", "|") \
    .schema(schema) \
    .load("/Volumes/initialcatalog/initialschema/initialvolume/streamfolder")\
    .withColumn("Age", lit(22)).select("ID", "NAME", "Age").writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", chkpointpath) \
    .trigger(availableNow=True)\
    .toTable("initialcatalog.initialschema.student_stream")






In [0]:
%sql
USE CATALOG initialcatalog;
USE SCHEMA initialschema;
--drop table student_stream;
create table if not exists student_stream (ID int, NAME string, Age int);
select * from student_stream;

In [0]:
#Databricks provided random stream generator
file_path = "/databricks-datasets/structured-streaming/events"
checkpoint_path = "/tmp/ss-tutorial/_checkpoint"

raw_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .load(file_path)
)

In [0]:
display(raw_df)

In [0]:
%sql
